# Argo Monte Carlo Simulation - Complete Tutorial

**Version:** 5.0.0-alpha.1  
**Author:** Argo Development Team  
**Date:** October 2025

Welcome to the Argo Monte Carlo Simulation tutorial! This notebook demonstrates all features of the argo-core library.

## What is Argo?

Argo is a modern Monte Carlo simulation engine for risk analysis and uncertainty quantification. It provides:

- **14 Probability Distributions** (10 continuous + 4 discrete)
- **30 Statistical Functions** (descriptive stats, percentiles, confidence intervals, risk metrics, distribution fitting)
- **Monte Carlo Simulation Engine** with correlation support
- **Formula Evaluation** with dependency resolution
- **Reproducible Results** with seeded random number generation

## Prerequisites

This notebook requires Node.js and the `tslab` package to run TypeScript in Jupyter.

### Installation

```bash
# Install dependencies (from project root)
cd /path/to/argo
npm install

# Install tslab globally (if not already installed)
npm install -g tslab

# Register tslab with Jupyter
tslab install --python=python3

# Start Jupyter
jupyter notebook
```

---

## 1. Setup and Imports

First, let's import the Argo library. Since we're in the monorepo, we'll import from the local package.

In [ ]:
// Import from local argo-core package using require (works better with tslab)
const argo = require('../packages/argo-core/dist/index');

// Destructure the exports
const {
  // Distributions
  NormalDistribution,
  UniformDistribution,
  TriangularDistribution,
  LogNormalDistribution,
  ExponentialDistribution,
  BetaDistribution,
  GammaDistribution,
  WeibullDistribution,
  ParetoDistribution,
  PERTDistribution,
  BinomialDistribution,
  PoissonDistribution,
  GeometricDistribution,
  HypergeometricDistribution,
  
  // Utilities
  SimpleRNG,
  
  // Statistical Functions
  mean,
  median,
  standardDeviation,
  percentile,
  quartiles,
  confidenceIntervalNormal,
  valueAtRisk,
  conditionalVaR,
  fitNormal,
  
  // Simulation Engine
  MonteCarloEngine
} = argo;

console.log('✅ Argo library loaded successfully!');

---

## 2. Probability Distributions

Argo provides 14 probability distributions. Let's explore the most commonly used ones.

### 2.1 Normal Distribution

The normal (Gaussian) distribution is the most common continuous distribution.

In [ ]:
// Create a normal distribution with mean=100, stddev=15
const normal = new NormalDistribution(100, 15);

// Create a seeded RNG for reproducibility
const rng = new SimpleRNG(42);

// Generate 10,000 samples
const samples = [];
for (let i = 0; i < 10000; i++) {
  samples.push(normal.sample(rng));
}

console.log('Normal Distribution (μ=100, σ=15):');
console.log(`  Theoretical mean: ${normal.mean}`);
console.log(`  Sample mean: ${mean(samples).toFixed(2)}`);
console.log(`  Theoretical std dev: ${normal.stddev}`);
console.log(`  Sample std dev: ${standardDeviation(samples).toFixed(2)}`);
console.log(`  Median: ${median(samples).toFixed(2)}`);
console.log(`  95th percentile: ${percentile(samples, 95).toFixed(2)}`);

### 2.2 Triangular Distribution

The triangular distribution is popular for modeling expert estimates (min, most likely, max).

In [ ]:
// Create triangular distribution: min=50, mode=100, max=150
const triangular = new TriangularDistribution(50, 100, 150);
const rng2 = new SimpleRNG(123);

const triSamples = [];
for (let i = 0; i < 10000; i++) {
  triSamples.push(triangular.sample(rng2));
}

console.log('\nTriangular Distribution (min=50, mode=100, max=150):');
console.log(`  Mean: ${mean(triSamples).toFixed(2)}`);
console.log(`  Median: ${median(triSamples).toFixed(2)}`);
console.log(`  Std Dev: ${standardDeviation(triSamples).toFixed(2)}`);

const q = quartiles(triSamples);
console.log(`  Quartiles: Q1=${q.q1.toFixed(2)}, Q2=${q.q2.toFixed(2)}, Q3=${q.q3.toFixed(2)}`);

### 2.3 PERT Distribution

The PERT distribution is similar to triangular but smoother, commonly used in project management.

In [ ]:
// Create PERT distribution: min=10, mode=20, max=40
const pert = new PERTDistribution(10, 20, 40);
const rng3 = new SimpleRNG(456);

const pertSamples = [];
for (let i = 0; i < 10000; i++) {
  pertSamples.push(pert.sample(rng3));
}

console.log('\nPERT Distribution (min=10, mode=20, max=40):');
console.log(`  Mean: ${mean(pertSamples).toFixed(2)}`);
console.log(`  Median: ${median(pertSamples).toFixed(2)}`);
console.log(`  90% Confidence Interval: ${confidenceIntervalNormal(pertSamples, 0.9).lower.toFixed(2)} - ${confidenceIntervalNormal(pertSamples, 0.9).upper.toFixed(2)}`);

### 2.4 Discrete Distributions

Argo also supports discrete distributions for modeling countable events.

In [ ]:
// Binomial: number of successes in n trials
const binomial = new BinomialDistribution(20, 0.3);
const rng4 = new SimpleRNG(789);

const binomialSamples = [];
for (let i = 0; i < 10000; i++) {
  binomialSamples.push(binomial.sample(rng4));
}

console.log('\nBinomial Distribution (n=20, p=0.3):');
console.log(`  Mean: ${mean(binomialSamples).toFixed(2)} (expected: ${binomial.mean})`);

// Poisson: number of events in fixed interval
const poisson = new PoissonDistribution(5);
const rng5 = new SimpleRNG(999);

const poissonSamples = [];
for (let i = 0; i < 10000; i++) {
  poissonSamples.push(poisson.sample(rng5));
}

console.log('\nPoisson Distribution (λ=5):');
console.log(`  Mean: ${mean(poissonSamples).toFixed(2)} (expected: ${poisson.mean})`);

---

## 3. Statistical Analysis

Argo provides 30 statistical functions for analyzing simulation results.

### 3.1 Risk Metrics

Value at Risk (VaR) and Conditional VaR are critical for risk analysis.

In [ ]:
// Generate project cost data (normally distributed around $1M with $200k std dev)
const costDist = new NormalDistribution(1000000, 200000);
const rng6 = new SimpleRNG(111);

const projectCosts = [];
for (let i = 0; i < 10000; i++) {
  projectCosts.push(costDist.sample(rng6));
}

console.log('\nProject Cost Risk Analysis:');
console.log(`  Mean cost: $${(mean(projectCosts) / 1000).toFixed(0)}k`);
console.log(`  Median cost: $${(median(projectCosts) / 1000).toFixed(0)}k`);
console.log(`  Std deviation: $${(standardDeviation(projectCosts) / 1000).toFixed(0)}k`);
console.log('\nRisk Metrics:');
console.log(`  VaR (95%): $${(valueAtRisk(projectCosts, 0.95) / 1000).toFixed(0)}k (95% chance cost is below this)`);
console.log(`  CVaR (95%): $${(conditionalVaR(projectCosts, 0.95) / 1000).toFixed(0)}k (average of worst 5% outcomes)`);

const ci90 = confidenceIntervalNormal(projectCosts, 0.9);
console.log(`  90% Confidence Interval: $${(ci90.lower / 1000).toFixed(0)}k - $${(ci90.upper / 1000).toFixed(0)}k`);

### 3.2 Distribution Fitting

Fit distributions to historical data.

In [ ]:
// We have historical data - let's fit a distribution to it
const historicalData = projectCosts;

const fitted = fitNormal(historicalData);

console.log('\nFitted Normal Distribution:');
console.log(`  μ (mean): $${(fitted.mu / 1000).toFixed(0)}k`);
console.log(`  σ (std dev): $${(fitted.sigma / 1000).toFixed(0)}k`);

// Create a new distribution from fitted parameters
const fittedDist = new NormalDistribution(fitted.mu, fitted.sigma);
console.log(`  Fitted distribution mean: $${(fittedDist.mean / 1000).toFixed(0)}k`);

---

## 4. Monte Carlo Simulation Engine

The MonteCarloEngine is the heart of Argo. It allows you to:
- Define input variables with probability distributions
- Define formula variables that calculate based on inputs
- Run thousands of iterations to analyze outcomes
- Support correlated variables

### 4.1 Simple Simulation

Let's simulate a simple profit calculation.

In [ ]:
// Create simulation engine
const engine = new MonteCarloEngine(new SimpleRNG(42));

// Define simulation: Profit = Revenue - Cost
const simpleConfig = {
  iterations: 10000,
  variables: [
    {
      name: 'Revenue',
      type: 'input' as const,
      distribution: new NormalDistribution(1000000, 150000)
    },
    {
      name: 'Cost',
      type: 'input' as const,
      distribution: new NormalDistribution(700000, 100000)
    },
    {
      name: 'Profit',
      type: 'formula' as const,
      formula: 'Revenue - Cost'
    }
  ]
};

// Run simulation
console.log('\nRunning simulation (10,000 iterations)...');
const result = engine.simulate(simpleConfig);

console.log('\nSimulation Results:');
console.log(`  Iterations: ${result.iterations}`);

console.log('\nRevenue:');
console.log(`  Mean: $${(result.statistics.Revenue.mean / 1000).toFixed(0)}k`);
console.log(`  Std Dev: $${(result.statistics.Revenue.stdDev / 1000).toFixed(0)}k`);

console.log('\nCost:');
console.log(`  Mean: $${(result.statistics.Cost.mean / 1000).toFixed(0)}k`);
console.log(`  Std Dev: $${(result.statistics.Cost.stdDev / 1000).toFixed(0)}k`);

console.log('\nProfit:');
console.log(`  Mean: $${(result.statistics.Profit.mean / 1000).toFixed(0)}k`);
console.log(`  Std Dev: $${(result.statistics.Profit.stdDev / 1000).toFixed(0)}k`);
console.log(`  Minimum: $${(result.statistics.Profit.min / 1000).toFixed(0)}k`);
console.log(`  Maximum: $${(result.statistics.Profit.max / 1000).toFixed(0)}k`);

// Calculate risk metrics
const profitSamples = result.samples.Profit;
console.log('\nProfit Risk Analysis:');
console.log(`  VaR (95%): $${(valueAtRisk(profitSamples, 0.95) / 1000).toFixed(0)}k`);
console.log(`  5th percentile: $${(percentile(profitSamples, 5) / 1000).toFixed(0)}k`);
console.log(`  95th percentile: $${(percentile(profitSamples, 95) / 1000).toFixed(0)}k`);

### 4.2 Complex Simulation with Dependencies

Let's model a more complex scenario: a construction project with multiple dependencies.

In [ ]:
// Construction project simulation
const projectConfig = {
  iterations: 10000,
  variables: [
    // Input variables (uncertainties)
    {
      name: 'MaterialCost',
      type: 'input' as const,
      distribution: new TriangularDistribution(400000, 500000, 700000)
    },
    {
      name: 'LaborDays',
      type: 'input' as const,
      distribution: new PERTDistribution(100, 120, 180)
    },
    {
      name: 'LaborRatePerDay',
      type: 'input' as const,
      distribution: new NormalDistribution(2000, 200)
    },
    {
      name: 'EquipmentCost',
      type: 'input' as const,
      distribution: new UniformDistribution(50000, 100000)
    },
    
    // Calculated variables (formulas)
    {
      name: 'LaborCost',
      type: 'formula' as const,
      formula: 'LaborDays * LaborRatePerDay'
    },
    {
      name: 'DirectCosts',
      type: 'formula' as const,
      formula: 'MaterialCost + LaborCost + EquipmentCost'
    },
    {
      name: 'Overhead',
      type: 'formula' as const,
      formula: 'DirectCosts * 0.15'  // 15% overhead
    },
    {
      name: 'TotalCost',
      type: 'formula' as const,
      formula: 'DirectCosts + Overhead'
    }
  ]
};

const projectEngine = new MonteCarloEngine(new SimpleRNG(2024));

console.log('\nRunning construction project simulation...');
const projectResult = projectEngine.simulate(projectConfig);

console.log('\nConstruction Project Results:');
console.log('\nInput Uncertainties:');
console.log(`  Material Cost: $${(projectResult.statistics.MaterialCost.mean / 1000).toFixed(0)}k ± $${(projectResult.statistics.MaterialCost.stdDev / 1000).toFixed(0)}k`);
console.log(`  Labor Days: ${projectResult.statistics.LaborDays.mean.toFixed(1)} ± ${projectResult.statistics.LaborDays.stdDev.toFixed(1)} days`);
console.log(`  Labor Rate: $${projectResult.statistics.LaborRatePerDay.mean.toFixed(0)}/day`);

console.log('\nTotal Cost Analysis:');
console.log(`  Mean: $${(projectResult.statistics.TotalCost.mean / 1000).toFixed(0)}k`);
console.log(`  Std Dev: $${(projectResult.statistics.TotalCost.stdDev / 1000).toFixed(0)}k`);
console.log(`  Min: $${(projectResult.statistics.TotalCost.min / 1000).toFixed(0)}k`);
console.log(`  Max: $${(projectResult.statistics.TotalCost.max / 1000).toFixed(0)}k`);

const totalCostSamples = projectResult.samples.TotalCost;
console.log('\nCost Percentiles:');
console.log(`  P10: $${(percentile(totalCostSamples, 10) / 1000).toFixed(0)}k (10% chance below this)`);
console.log(`  P50: $${(percentile(totalCostSamples, 50) / 1000).toFixed(0)}k (median)`);
console.log(`  P90: $${(percentile(totalCostSamples, 90) / 1000).toFixed(0)}k (90% chance below this)`);

const budget = 900000;
const overBudget = totalCostSamples.filter(c => c > budget).length;
const probOverBudget = (overBudget / totalCostSamples.length) * 100;
console.log(`\nBudget Risk ($${budget / 1000}k budget):`);
console.log(`  Probability of exceeding budget: ${probOverBudget.toFixed(1)}%`);

### 4.3 Correlated Variables

Real-world variables are often correlated. Argo supports correlation using Gaussian copulas.

In [ ]:
// Model oil price and shipping cost (positively correlated)
const correlatedConfig = {
  iterations: 10000,
  variables: [
    {
      name: 'OilPrice',
      type: 'input' as const,
      distribution: new NormalDistribution(80, 15)  // $/barrel
    },
    {
      name: 'ShippingCost',
      type: 'input' as const,
      distribution: new NormalDistribution(5000, 1000)  // $
    },
    {
      name: 'Volume',
      type: 'input' as const,
      distribution: new UniformDistribution(1000, 2000)  // barrels
    },
    {
      name: 'TotalCost',
      type: 'formula' as const,
      formula: 'OilPrice * Volume + ShippingCost'
    }
  ],
  // Oil price and shipping cost are correlated (correlation = 0.7)
  correlations: {
    'OilPrice-ShippingCost': 0.7
  }
};

const corrEngine = new MonteCarloEngine(new SimpleRNG(3030));

console.log('\nRunning correlated simulation...');
const corrResult = corrEngine.simulate(correlatedConfig);

console.log('\nCorrelated Oil & Shipping Analysis:');
console.log(`  Oil Price: $${corrResult.statistics.OilPrice.mean.toFixed(2)}/barrel ± $${corrResult.statistics.OilPrice.stdDev.toFixed(2)}`);
console.log(`  Shipping Cost: $${(corrResult.statistics.ShippingCost.mean / 1000).toFixed(2)}k ± $${(corrResult.statistics.ShippingCost.stdDev / 1000).toFixed(2)}k`);

// Verify correlation
const oilSamples = corrResult.samples.OilPrice;
const shipSamples = corrResult.samples.ShippingCost;

// Calculate Pearson correlation
const oilMean = mean(oilSamples);
const shipMean = mean(shipSamples);
const oilStd = standardDeviation(oilSamples);
const shipStd = standardDeviation(shipSamples);

let correlation = 0;
for (let i = 0; i < oilSamples.length; i++) {
  correlation += ((oilSamples[i] - oilMean) / oilStd) * ((shipSamples[i] - shipMean) / shipStd);
}
correlation /= oilSamples.length;

console.log(`\nCorrelation Analysis:`);
console.log(`  Specified correlation: 0.70`);
console.log(`  Observed correlation: ${correlation.toFixed(2)}`);

console.log(`\nTotal Cost:`);
console.log(`  Mean: $${(corrResult.statistics.TotalCost.mean / 1000).toFixed(0)}k`);
console.log(`  Std Dev: $${(corrResult.statistics.TotalCost.stdDev / 1000).toFixed(0)}k`);

### 4.4 Progress Reporting

For long-running simulations, you can track progress with a callback.

In [ ]:
// Large simulation with progress tracking
const largeConfig = {
  iterations: 50000,
  variables: [
    {
      name: 'X',
      type: 'input' as const,
      distribution: new NormalDistribution(0, 1)
    },
    {
      name: 'Y',
      type: 'input' as const,
      distribution: new NormalDistribution(0, 1)
    },
    {
      name: 'Z',
      type: 'formula' as const,
      formula: 'sqrt(X*X + Y*Y)'
    }
  ]
};

const progressEngine = new MonteCarloEngine(new SimpleRNG(5000));

console.log('\nRunning large simulation (50,000 iterations)...');

const startTime = Date.now();

const largeResult = progressEngine.simulate(largeConfig, (iteration, total) => {
  const pct = ((iteration + 1) / total * 100).toFixed(0);
  if ((iteration + 1) % 10000 === 0) {
    console.log(`  Progress: ${pct}% (${iteration + 1}/${total})`);
  }
});

const elapsed = Date.now() - startTime;

console.log('\nLarge Simulation Complete:');
console.log(`  Iterations: ${largeResult.iterationsCompleted.toLocaleString()}`);
console.log(`  Time: ${elapsed}ms`);
console.log(`  Performance: ${(largeResult.iterationsCompleted / (elapsed / 1000)).toLocaleString(undefined, {maximumFractionDigits: 0})} iterations/second`);
console.log(`\nResults (Z = sqrt(X² + Y²)):`);
console.log(`  Mean: ${largeResult.statistics.Z.mean.toFixed(4)}`);
console.log(`  Std Dev: ${largeResult.statistics.Z.stdDev.toFixed(4)}`);

// Z follows a Rayleigh distribution with expected mean ≈ 1.253
console.log(`  Expected mean (Rayleigh): 1.2533`);

---

## 5. Real-World Example: Software Project Risk

Let's model a complete software development project with multiple uncertainties.

In [ ]:
// Software development project
const softwareConfig = {
  iterations: 10000,
  variables: [
    // Development effort (story points)
    {
      name: 'DevelopmentEffort',
      type: 'input' as const,
      distribution: new PERTDistribution(80, 120, 200)  // story points
    },
    
    // Team velocity (points per sprint)
    {
      name: 'TeamVelocity',
      type: 'input' as const,
      distribution: new NormalDistribution(25, 5)  // points/sprint
    },
    
    // Number of sprints
    {
      name: 'Sprints',
      type: 'formula' as const,
      formula: 'ceil(DevelopmentEffort / TeamVelocity)'
    },
    
    // Development duration (2-week sprints)
    {
      name: 'DurationWeeks',
      type: 'formula' as const,
      formula: 'Sprints * 2'
    },
    
    // Defects found (Poisson distribution)
    {
      name: 'Defects',
      type: 'input' as const,
      distribution: new PoissonDistribution(15)  // average 15 defects
    },
    
    // Time to fix defects (hours per defect)
    {
      name: 'HoursPerDefect',
      type: 'input' as const,
      distribution: new TriangularDistribution(2, 4, 12)
    },
    
    // Total defect fixing time
    {
      name: 'DefectFixingHours',
      type: 'formula' as const,
      formula: 'Defects * HoursPerDefect'
    },
    
    // Cost (assuming $150/hour blended rate)
    {
      name: 'DevelopmentCost',
      type: 'formula' as const,
      formula: 'DevelopmentEffort * 8 * 150'  // 8 hours per point
    },
    {
      name: 'DefectCost',
      type: 'formula' as const,
      formula: 'DefectFixingHours * 150'
    },
    {
      name: 'TotalCost',
      type: 'formula' as const,
      formula: 'DevelopmentCost + DefectCost'
    }
  ]
};

const swEngine = new MonteCarloEngine(new SimpleRNG(2025));

console.log('\n=== SOFTWARE PROJECT RISK ANALYSIS ===\n');
const swResult = swEngine.simulate(softwareConfig);

console.log('Development Effort:');
console.log(`  Mean: ${swResult.statistics.DevelopmentEffort.mean.toFixed(1)} story points`);
console.log(`  Range: ${swResult.statistics.DevelopmentEffort.min.toFixed(0)} - ${swResult.statistics.DevelopmentEffort.max.toFixed(0)} points`);

console.log('\nTimeline:');
console.log(`  Mean duration: ${swResult.statistics.DurationWeeks.mean.toFixed(1)} weeks (${(swResult.statistics.DurationWeeks.mean / 4.33).toFixed(1)} months)`);
console.log(`  Std dev: ${swResult.statistics.DurationWeeks.stdDev.toFixed(1)} weeks`);

const durationSamples = swResult.samples.DurationWeeks;
console.log(`  P50 (median): ${percentile(durationSamples, 50).toFixed(0)} weeks`);
console.log(`  P80: ${percentile(durationSamples, 80).toFixed(0)} weeks (80% confident)`);
console.log(`  P90: ${percentile(durationSamples, 90).toFixed(0)} weeks (90% confident)`);

console.log('\nDefects:');
console.log(`  Mean defects: ${swResult.statistics.Defects.mean.toFixed(1)}`);
console.log(`  Mean fixing time: ${swResult.statistics.DefectFixingHours.mean.toFixed(1)} hours`);

console.log('\nCost Analysis:');
console.log(`  Development cost: $${(swResult.statistics.DevelopmentCost.mean / 1000).toFixed(0)}k ± $${(swResult.statistics.DevelopmentCost.stdDev / 1000).toFixed(0)}k`);
console.log(`  Defect cost: $${(swResult.statistics.DefectCost.mean / 1000).toFixed(0)}k ± $${(swResult.statistics.DefectCost.stdDev / 1000).toFixed(0)}k`);
console.log(`  Total cost: $${(swResult.statistics.TotalCost.mean / 1000).toFixed(0)}k ± $${(swResult.statistics.TotalCost.stdDev / 1000).toFixed(0)}k`);

const costSamples = swResult.samples.TotalCost;
console.log('\nCost Percentiles:');
console.log(`  P10: $${(percentile(costSamples, 10) / 1000).toFixed(0)}k (optimistic)`);
console.log(`  P50: $${(percentile(costSamples, 50) / 1000).toFixed(0)}k (median)`);
console.log(`  P90: $${(percentile(costSamples, 90) / 1000).toFixed(0)}k (pessimistic)`);

const ci95 = confidenceIntervalNormal(costSamples, 0.95);
console.log(`  95% CI: $${(ci95.lower / 1000).toFixed(0)}k - $${(ci95.upper / 1000).toFixed(0)}k`);

// Budget scenarios
const budgets = [150000, 175000, 200000];
console.log('\nBudget Scenarios:');
for (const budget of budgets) {
  const withinBudget = costSamples.filter(c => c <= budget).length;
  const probability = (withinBudget / costSamples.length) * 100;
  console.log(`  $${budget / 1000}k budget: ${probability.toFixed(1)}% probability of staying within budget`);
}

---

## 6. Summary and Next Steps

### What We've Covered

✅ **14 Probability Distributions** - Normal, Uniform, Triangular, PERT, Beta, Gamma, and more  
✅ **30 Statistical Functions** - Descriptive stats, percentiles, confidence intervals, risk metrics  
✅ **Monte Carlo Simulation** - Single/multiple variables, formulas, dependencies  
✅ **Correlation Support** - Model correlated uncertainties with Gaussian copulas  
✅ **Real-World Examples** - Construction projects, software development, cost analysis  

### Performance

- 10,000 iteration simulation: ~10ms
- 50,000 iteration simulation: ~50ms
- Performance: 1M+ iterations/second

### Available Distributions

**Continuous:**
- Normal, Uniform, Triangular, PERT
- LogNormal, Exponential, Beta, Gamma
- Weibull, Pareto

**Discrete:**
- Binomial, Poisson, Geometric, Hypergeometric

### Next Steps

1. **Explore More Distributions** - Try Beta, Gamma, Weibull for different scenarios
2. **Build Your Model** - Adapt the examples to your specific use case
3. **Add Correlations** - Model realistic dependencies between variables
4. **Analyze Results** - Use VaR, CVaR, and percentiles for decision-making
5. **Scale Up** - Run 100k+ iterations for high-precision analysis

### Resources

- **Documentation:** `/docs/` directory in the repo
- **Tests:** `/packages/argo-core/tests/` for more examples
- **Source Code:** `/packages/argo-core/src/`

### Coming Soon

- CLI tool for running simulations from command line
- Excel add-in for Office 365
- Visualization tools
- More statistical functions

---

**Thank you for using Argo!**

For questions, issues, or contributions, visit: https://github.com/montge/argo